# 00_Data_Acquisition — CFPB Narratives Archive

**Objectif:** Load & validate source files, manifests and integrity of the frozen datasets

In [1]:
import os
if 'notebooks' in os.getcwd():
    os.chdir('..')
os.makedirs('output', exist_ok=True)
print(f'Working directory: {os.getcwd()}')

Working directory: D:\Data analyst\Portfolio\Banque\bank-complaint-routing-nlp


In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.width', 200)
print('✓ Libraries loaded')

✓ Libraries loaded


In [3]:
probe = pd.read_csv('reports/source_probe.csv')
inv = pd.read_csv('reports/local_inventory.csv')
print(f'✓ Source URLs probed: {len(probe)} (HTTP 200: {(probe["http_status"].astype(str) == "200").sum()})')
print(f'✓ Local CSV files: {int(inv["present"].sum())}/21 — {inv.loc[inv["present"], "size_mb"].astype(float).sum() / 1024:.2f} GB')
sel = inv[inv['selected']]
print(f'Selected exports (2026): {sel["id"].tolist()} — {sel["size_mb"].astype(float).sum() / 1024:.2f} GB')

✓ Source URLs probed: 21 (HTTP 200: 21)
✓ Local CSV files: 21/21 — 8.66 GB
Selected exports (2026): [15, 16, 17, 18, 19, 20, 21] — 1.53 GB


## 1. Selected Exports

In [4]:
print(sel[['id', 'file', 'size_mb', 'detected_format', 'n_header_fields']].to_string(index=False))

 id                                                  file  size_mb detected_format  n_header_fields
 15 CCDB_Export_15_January_2026_through_February_2026.csv    349.5             csv               16
 16                         CCDB_Export_16_March_2026.csv    209.4             csv               16
 17                         CCDB_Export_17_April_2026.csv    213.1             csv               16
 18                           CCDB_Export_18_May_2026.csv    209.4             csv               16
 19                          CCDB_Export_19_June_2026.csv    209.7             csv               16
 20                          CCDB_Export_20_July_2026.csv    206.2             csv               16
 21                        CCDB_Export_21_August_2026.csv    169.0             csv               16


## 2. Schema

In [5]:
cols = json.load(open('data/interim/columns.json', encoding='utf-8'))
print('Original -> normalised column names:')
for o, n in cols['original_to_normalized'].items():
    print(f'  {o:<35} -> {n}')
print('\nKey columns:', cols['keys'])
print(f"Rows loaded (2026): {cols['n_rows_all']:,} | with narrative: {cols['n_rows_with_narrative']:,}")

Original -> normalised column names:
  Date received                       -> date_received
  Product                             -> product
  Sub-product                         -> sub_product
  Issue                               -> issue
  Sub-issue                           -> sub_issue
  Consumer complaint narrative        -> consumer_complaint_narrative
  Company public response             -> company_public_response
  Company                             -> company
  State                               -> state
  ZIP code                            -> zip_code
  Tags                                -> tags
  Submitted via                       -> submitted_via
  Date sent to company                -> date_sent_to_company
  Company response to consumer        -> company_response_to_consumer
  Timely response?                    -> timely_response
  Complaint ID                        -> complaint_id

Key columns: {'complaint_id': 'complaint_id', 'date_received': 'date_received', 'p

## 3. Integrity of the Frozen Datasets (sha256)

In [6]:
import hashlib
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(8 * 1024**2), b''):
            h.update(b)
    return h.hexdigest()

man = json.load(open('data/processed/MANIFEST.json', encoding='utf-8'))
for name in ('all', 'train', 'val', 'test'):
    ok = sha256(f"data/processed/{man[name]['file']}") == man[name]['sha256']
    print(f"{'✓' if ok else '✗'} {man[name]['file']:<28} rows={man[name]['rows']:>7,}  sha256 {'OK' if ok else 'MISMATCH'}")

✓ complaints_clean.parquet     rows=106,307  sha256 OK


✓ train.parquet                rows= 74,415  sha256 OK
✓ val.parquet                  rows= 15,945  sha256 OK
✓ test.parquet                 rows= 15,947  sha256 OK


## 4. CSV Reading Check (strict sequential mode vs Phase 1)

In [7]:
print(open('reports/csv_parsing_check.md', encoding='utf-8').read())

# Vérification de la lecture CSV des exports 2026

Relecture des exports sélectionnés en mode séquentiel strict (`header=true, all_varchar=true, parallel=false, quote='"', escape='"', max_line_size=20000000`) et comparaison avec la sortie de la Phase 1 (`data/interim/complaints_all.parquet`).

| export_id | rows_phase1 | rows_strict | ids_only_phase1 | ids_only_strict | pair_hashes_differing | status |
|---|---|---|---|---|---|---|
| 15 | 1038383 | 1038383 | 0 | 0 | 0 | OK |
| 16 | 614945 | 614945 | 0 | 0 | 0 | OK |
| 17 | 618293 | 618293 | 0 | 0 | 0 | OK |
| 18 | 619940 | 619940 | 0 | 0 | 0 | OK |
| 19 | 620794 | 620794 | 0 | 0 | 0 | OK |
| 20 | 672036 | 672036 | 0 | 0 | 0 | OK |
| 21 | 667140 | 667140 | 0 | 0 | 0 | OK |

**Conclusion : lectures identiques — les jeux de la Phase 2 restent valides.**

